<a href="https://colab.research.google.com/github/PoojithaUndavalli/Intelligent-Elevator-Control-System-Using-Fuzzy-Logic/blob/main/SC_CaseStudy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install numpy scikit-fuzzy matplotlib pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 8.6 MB/s  0:00:00


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import skfuzzy as fuzz
from skfuzzy import control as ctrl

import random
import time

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
# Building Configuration

TOTAL_FLOORS = 10
TOTAL_ELEVATORS = 4
MAX_CAPACITY = 10

# Initial elevator positions
initial_positions = [1, 4, 7, 10]

print("INTELLIGENT ELEVATOR CONTROL SYSTEM")
print("-----------------------------------")

print("Total Floors:", TOTAL_FLOORS)
print("Total Elevators:", TOTAL_ELEVATORS)
print("Maximum Capacity:", MAX_CAPACITY)

for i, floor in enumerate(initial_positions):
    print(f"Elevator E{i+1} - Floor {floor}")

INTELLIGENT ELEVATOR CONTROL SYSTEM
-----------------------------------
Total Floors: 10
Total Elevators: 4
Maximum Capacity: 10
Elevator E1 - Floor 1
Elevator E2 - Floor 4
Elevator E3 - Floor 7
Elevator E4 - Floor 10


In [ ]:

import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
from itertools import product

# ---------------------------------------
# FUZZY INPUT VARIABLES
# ---------------------------------------

distance = ctrl.Antecedent(np.arange(0, 10, 0.1), 'distance')
direction = ctrl.Antecedent(np.arange(0, 1.01, 0.01), 'direction')
load = ctrl.Antecedent(np.arange(0, 101, 1), 'load')

# ---------------------------------------
# FUZZY OUTPUT VARIABLE
# ---------------------------------------

priority = ctrl.Consequent(
    np.arange(0, 101, 1),
    'priority'
)

# ---------------------------------------
# MEMBERSHIP FUNCTIONS: DISTANCE
# ---------------------------------------

distance['near'] = fuzz.trimf(distance.universe, [0, 0, 3])
distance['medium'] = fuzz.trimf(distance.universe, [1, 4.5, 8])
distance['far'] = fuzz.trimf(distance.universe, [6, 9, 9.9])

# ---------------------------------------
# MEMBERSHIP FUNCTIONS: DIRECTION
# ---------------------------------------

direction['poor'] = fuzz.trimf(direction.universe, [0, 0, 0.5])
direction['moderate'] = fuzz.trimf(direction.universe, [0.25, 0.5, 0.75])
direction['good'] = fuzz.trimf(direction.universe, [0.5, 1, 1])

# ---------------------------------------
# MEMBERSHIP FUNCTIONS: LOAD
# ---------------------------------------

load['low'] = fuzz.trimf(load.universe, [0, 0, 40])
load['medium'] = fuzz.trimf(load.universe, [20, 50, 80])
load['high'] = fuzz.trimf(load.universe, [60, 100, 100])

# ---------------------------------------
# MEMBERSHIP FUNCTIONS: PRIORITY
# ---------------------------------------

priority['very_low'] = fuzz.trimf(priority.universe, [0, 0, 25])
priority['low'] = fuzz.trimf(priority.universe, [10, 30, 50])
priority['medium'] = fuzz.trimf(priority.universe, [35, 50, 65])
priority['high'] = fuzz.trimf(priority.universe, [50, 70, 90])
priority['very_high'] = fuzz.trimf(priority.universe, [75, 100, 100])

print("Fuzzy variables and membership functions created successfully!")


Fuzzy variables and membership functions created successfully!


In [ ]:

# ---------------------------------------
# FUZZY RULE BASE
# ---------------------------------------

distance_labels = ['near', 'medium', 'far']
direction_labels = ['poor', 'moderate', 'good']
load_labels = ['low', 'medium', 'high']

# Base values represent the relative contribution
distance_score = {
    'near': 2,
    'medium': 1,
    'far': 0
}

direction_score = {
    'poor': 0,
    'moderate': 1,
    'good': 2
}

load_score = {
    'low': 2,
    'medium': 1,
    'high': 0
}

priority_labels = [
    'very_low',
    'low',
    'medium',
    'high',
    'very_high'
]

rules = []

for d, dr, l in product(
    distance_labels,
    direction_labels,
    load_labels
):
    total_score = (
        distance_score[d]
        + direction_score[dr]
        + load_score[l]
    )

    if total_score <= 1:
        result = 'very_low'
    elif total_score <= 2:
        result = 'low'
    elif total_score <= 3:
        result = 'medium'
    elif total_score <= 4:
        result = 'high'
    else:
        result = 'very_high'

    rule = ctrl.Rule(
        distance[d] & direction[dr] & load[l],
        priority[result]
    )

    rules.append(rule)

print("Total fuzzy rules created:", len(rules))

# Create the fuzzy control system
elevator_control = ctrl.ControlSystem(rules)

print("Fuzzy Control System created successfully!")


Total fuzzy rules created: 27
Fuzzy Control System created successfully!


In [ ]:

def calculate_priority(elevator_floor, passenger_floor,
                       elevator_direction, passenger_direction,
                       current_load):

    # Distance between elevator and passenger
    dist = abs(elevator_floor - passenger_floor)

    # Direction compatibility
    if elevator_direction == "IDLE":
        direction_value = 0.75

    elif elevator_direction == passenger_direction:
        # Elevator must be moving towards the passenger
        if passenger_direction == "UP" and elevator_floor <= passenger_floor:
            direction_value = 1.0
        elif passenger_direction == "DOWN" and elevator_floor >= passenger_floor:
            direction_value = 1.0
        else:
            direction_value = 0.1

    else:
        direction_value = 0.1

    # Load percentage
    load_percentage = (current_load / MAX_CAPACITY) * 100

    # Create simulation
    simulation = ctrl.ControlSystemSimulation(elevator_control)

    simulation.input['distance'] = dist
    simulation.input['direction'] = direction_value
    simulation.input['load'] = load_percentage

    simulation.compute()

    return simulation.output['priority']


print("Priority calculation function created successfully!")


Priority calculation function created successfully!


In [ ]:

# Sample elevator information

elevators = [
    {
        'id': 'E1',
        'floor': 1,
        'direction': 'IDLE',
        'load': 2
    },
    {
        'id': 'E2',
        'floor': 4,
        'direction': 'UP',
        'load': 5
    },
    {
        'id': 'E3',
        'floor': 7,
        'direction': 'DOWN',
        'load': 3
    },
    {
        'id': 'E4',
        'floor': 10,
        'direction': 'IDLE',
        'load': 8
    }
]

# Passenger request
passenger_floor = 5
passenger_direction = "UP"

results = []

for elevator in elevators:

    score = calculate_priority(
        elevator['floor'],
        passenger_floor,
        elevator['direction'],
        passenger_direction,
        elevator['load']
    )

    results.append({
        'Elevator': elevator['id'],
        'Floor': elevator['floor'],
        'Direction': elevator['direction'],
        'Load': elevator['load'],
        'Priority Score': round(score, 2)
    })

results_df = pd.DataFrame(results)

print("PASSENGER REQUEST")
print("Floor:", passenger_floor)
print("Direction:", passenger_direction)

display(results_df)

# Select the elevator with highest priority
best_elevator = max(results, key=lambda x: x['Priority Score'])

print("\nSELECTED ELEVATOR:", best_elevator['Elevator'])
print("Priority Score:", best_elevator['Priority Score'])


PASSENGER REQUEST
Floor: 5
Direction: UP


,Elevator,Floor,Direction,Load,Priority Score
0,E1,1,IDLE,2,90.28
1,E2,4,UP,5,90.97
2,E3,7,DOWN,3,49.10
3,E4,10,IDLE,8,50.00



SELECTED ELEVATOR: E2
Priority Score: 90.97
